In [59]:
# imports
import pandas as pd
import altair as alt
import matplotlib.pyplot as plt

alt.data_transformers.disable_max_rows()

# load dataset

df.head()

,tconst,primaryTitle,startYear,endYear,rank,averageRating,numVotes,directors,writers,genres,IMDbLink,Title_IMDb_Link,era,primaryGenre,showDuration,durationLabel,main_genre
0,tt0903747,Breaking Bad,2008,2013.0,1,9.5,2593781,"Tricia Brock, Thomas Schnauz, Scott Winant, Ji...","George Mastras, Moira Walley-Beckett, John Shi...",Crime,"<a href=""https://www.imdb.com/title/tt0903747""...","<a href=""https://www.imdb.com/title/tt0903747""...",Pre-Streaming,Crime,5.0,5 years,Crime
0,tt0903747,Breaking Bad,2008,2013.0,1,9.5,2593781,"Tricia Brock, Thomas Schnauz, Scott Winant, Ji...","George Mastras, Moira Walley-Beckett, John Shi...",Drama,"<a href=""https://www.imdb.com/title/tt0903747""...","<a href=""https://www.imdb.com/title/tt0903747""...",Pre-Streaming,Drama,5.0,5 years,Drama
0,tt0903747,Breaking Bad,2008,2013.0,1,9.5,2593781,"Tricia Brock, Thomas Schnauz, Scott Winant, Ji...","George Mastras, Moira Walley-Beckett, John Shi...",Thriller,"<a href=""https://www.imdb.com/title/tt0903747""...","<a href=""https://www.imdb.com/title/tt0903747""...",Pre-Streaming,Thriller,5.0,5 years,Thriller
2,tt7366338,Chernobyl,2019,2019.0,3,9.3,1027105,Johan Renck,Craig Mazin,Drama,"<a href=""https://www.imdb.com/title/tt7366338""...","<a href=""https://www.imdb.com/title/tt7366338""...",Streaming (2010–2019),Drama,0.0,Less than 1 year,Drama
2,tt7366338,Chernobyl,2019,2019.0,3,9.3,1027105,Johan Renck,Craig Mazin,History,"<a href=""https://www.imdb.com/title/tt7366338""...","<a href=""https://www.imdb.com/title/tt7366338""...",Streaming (2010–2019),History,0.0,Less than 1 year,History


In [60]:
# cleaning the data
# convert columns 
df["startYear"] = pd.to_numeric(df["startYear"], errors="coerce")
df["endYear"] = pd.to_numeric(df["endYear"], errors="coerce")
df["averageRating"] = pd.to_numeric(df["averageRating"], errors="coerce")

# drop missing core fields
df = df.dropna(subset=["startYear", "averageRating", "genres"])

# classify eras
def classify_era(year):
    if year < 2010:
        return "Pre-Streaming"
    elif year < 2020:
        return "Streaming (2010–2019)"
    else:
        return "COVID Era (2020+)"

df["era"] = df["startYear"].apply(classify_era)

# genre cleaning
df["genres"] = df["genres"].str.split(",")
df = df.explode("genres")
df["genres"] = df["genres"].str.strip()

# primary genre
df["primaryGenre"] = df["genres"]

# show duration
df["showDuration"] = df["endYear"] - df["startYear"]

df["durationLabel"] = df.apply(
    lambda row: "Ongoing" if pd.isna(row["endYear"])
    else "Less than 1 year" if row["showDuration"] == 0
    else f"{int(row['showDuration'])} years",
    axis=1
)

# final clean
df = df.dropna(subset=["numVotes"])

In [61]:
# use filtered data
df_filtered = df[df['numVotes'] < 500000]

# main genre
df_filtered['main_genre'] = df_filtered['genres']

# scatter plot
scatter = alt.Chart(df_filtered).mark_circle(size=60).encode(
    x=alt.X('numVotes:Q', title='Number of Votes (Popularity)'),
    y=alt.Y('averageRating:Q', title='Average Rating'),
    color=alt.Color('main_genre:N', title='Genre'),
    
    tooltip=[
        alt.Tooltip('primaryTitle:N', title='Show'),
        alt.Tooltip('main_genre:N', title='Genre'),
        alt.Tooltip('averageRating:Q', title='Rating'),
        alt.Tooltip('numVotes:Q', title='Votes')
    ]
).properties(
    title='Popularity vs Average Rating by Genre',
    width=600,
    height=400
)

scatter
scatter.save("scatterplot.html")

/var/folders/4p/g_1nmc1d12q68cb0pp6trt1m0000gn/T/ipykernel_17947/2682414357.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered['main_genre'] = df_filtered['genres']


In [54]:
#imports and loading data
import altair as alt
alt.data_transformers.disable_max_rows()

# split genres into multiple rows
df['genres'] = df['genres'].str.split(',')
df = df.explode('genres')

# Remove extra spaces
df['genres'] = df['genres'].str.strip()

In [55]:
# clean data 
df['startYear'] = pd.to_numeric(df['startYear'], errors='coerce')

# split genres into multiple rows
df['genres'] = df['genres'].str.split(',')
df = df.explode('genres')

# filter years from only 2005-2025
df = df[(df['startYear'] >= 2005) & (df['startYear'] <= 2025)]

# group data
df_grouped = df.groupby(['startYear', 'genres'])['averageRating'].mean().reset_index()

# dropdown selection 
genre_dropdown = alt.binding_select(
    options=sorted(df_grouped['genres'].dropna().unique())
)

selection = alt.selection_point(
    fields=['genres'],
    bind=genre_dropdown,
    value=sorted(df_grouped['genres'].dropna().unique())[0]
)

#  allow large dataset
alt.data_transformers.disable_max_rows()

# chart 
chart = alt.Chart(df_grouped).mark_line(point=True).encode(
    x=alt.X(
        'startYear:Q',
        title='Year',
        axis=alt.Axis(labelAngle=-30, tickCount=8)
    ),
    y=alt.Y(
        'averageRating:Q',
        title='Average Rating'
    ),
    tooltip=['genres', 'startYear', 'averageRating']
).add_params(
    selection
).transform_filter(
    selection
).properties(
    title='Average Rating Over Time by Genre (2005–2025)',
    width=600,
    height=400
)

chart
chart.save("avg_ratings.html")

In [56]:
era_summary = (
    df.groupby("era", as_index=False)
    .agg(
        avg_rating=("averageRating", "mean"),
        show_count=("tconst", "count")
    )
)

era_order = ["Pre-Streaming", "Streaming (2010–2019)", "COVID Era (2020+)"]

hover = alt.selection_point(fields=["era"], on="mouseover", clear="mouseout")

bars = alt.Chart(era_summary).mark_bar(size=80).encode(
    x=alt.X("era:N", sort=era_order, title="Era"),
    y=alt.Y("avg_rating:Q", title="Average IMDb Rating", scale=alt.Scale(domain=[0, 10])),
    color=alt.condition(hover, alt.value("#4C78A8"), alt.value("#9ecae9")),
    tooltip=[
        "era:N",
        alt.Tooltip("avg_rating:Q", format=".2f"),
        "show_count:Q"
    ]
).add_params(hover)

labels = alt.Chart(era_summary).mark_text(dy=-10).encode(
    x=alt.X("era:N", sort=era_order),
    y="avg_rating:Q",
    text=alt.Text("avg_rating:Q", format=".2f")
)

viz1 = (bars + labels).properties(
    title="Average TV Show Ratings Across Eras",
    width=420,
    height=300
)

viz1

(bars + labels).save("era_bar_chart.html")


In [57]:
# use main genre (first only for cleaner heatmap)
df["main_genre"] = df["genres"]

# keep top genres
top_genres = df["main_genre"].value_counts().head(10).index
genre_df = df[df["main_genre"].isin(top_genres)]

# summarize
genre_summary = (
    genre_df.groupby(["era", "main_genre"], as_index=False)
    .agg(
        avg_rating=("averageRating", "mean"),
        show_count=("tconst", "count")
    )
)

select_genre = alt.selection_point(fields=["main_genre"])

heatmap = alt.Chart(genre_summary).mark_rect(stroke="white").encode(
    x=alt.X("era:N", sort=era_order, title="Era"),
    y=alt.Y("main_genre:N", title="Genre"),

    color=alt.Color(
        "avg_rating:Q",
        title="Average IMDb Rating",
        scale=alt.Scale(scheme="blues", domain=[7.0, 8.5])
    ),

    opacity=alt.condition(select_genre, alt.value(1), alt.value(0.25)),

    tooltip=[
        alt.Tooltip("main_genre:N", title="Genre"),
        alt.Tooltip("era:N", title="Era"),
        alt.Tooltip("avg_rating:Q", title="Avg Rating", format=".2f"),
        alt.Tooltip("show_count:Q", title="# Shows")
    ]
).add_params(select_genre).properties(
    title="Average IMDb Rating by Genre Across Television Eras",
    width=420,
    height=320
)

heatmap
heatmap.save("genre_heatmap.html")


In [58]:
# --- create a working copy ---
df_filtered = df.copy()

# --- calculate duration ---
df_filtered['showDuration'] = df_filtered['endYear'] - df_filtered['startYear']

# --- handle duration safely ---
df_filtered['durationLabel'] = df_filtered.apply(
    lambda row: "Ongoing" if pd.isna(row["endYear"])
    else "Less than 1 year" if row["showDuration"] == 0
    else f"{int(row['showDuration'])} years",
    axis=1
)

# --- drop missing values ---
df_filtered = df_filtered.dropna(subset=['startYear', 'numVotes', 'primaryGenre'])

# --- slider ---
rating_slider = alt.binding_range(min=0, max=10, step=0.5, name='Minimum Rating: ')
rating_selection = alt.param(name='min_rating', value=0, bind=rating_slider)

# --- chart ---
chart = alt.Chart(df_filtered).mark_circle(opacity=0.6).encode(
    x=alt.X(
        'startYear:Q',
        title='Start Year',
        scale=alt.Scale(zero=False),
        axis=alt.Axis(format='d')
    ),
    y=alt.Y(
        'showDuration:Q',
        title='Show Duration (Years)'
    ),
    size=alt.Size(
        'numVotes:Q',
        title='Number of Votes',
        scale=alt.Scale(range=[20, 1000])
    ),
    color=alt.Color(
        'primaryGenre:N',
        title='Genre',
        scale=alt.Scale(scheme='tableau10')
    ),
    tooltip=[
        alt.Tooltip('primaryTitle:N', title='Title'),
        alt.Tooltip('startYear:Q', title='Start Year', format='d'),
        alt.Tooltip('endYear:Q', title='End Year', format='d'),
        alt.Tooltip('durationLabel:N', title='Duration'),
        alt.Tooltip('averageRating:Q', title='Rating', format='.1f'),
        alt.Tooltip('numVotes:Q', title='Votes', format=','),
        alt.Tooltip('primaryGenre:N', title='Genre')
    ]
).transform_filter(
    alt.datum.averageRating >= rating_selection
).add_params(
    rating_selection
).properties(
    width=700,
    height=450,
    title='IMDb TV Show Longevity Bubble Chart'
).interactive()

chart
chart.save('plot.html')